# Emotional Pain Localizer — Functional ROI Definition

**Lab:** Saxe Lab, MIT  
**Task:** Emotional Pain (EP) vs. Physical Pain (PP) reading paradigm  
**Goal:** Define a per-participant functional ROI by contrasting EP > PP, restricted to an anatomical prior (Neurosynth "theory of mind" reverse-inference map). These masks feed into a separate main-task analysis.

---

## Experiment summary

Participants read short stories describing emotional pain (EP) or physical pain (PP).  
Each participant completed **5 EP + 5 PP trials**.  
fMRI data were preprocessed with **fMRIPrep**.

---

## How to use this notebook

Each section is **self-contained**: it reads inputs from disk and writes outputs to disk, so sections can be re-run independently. Run sections in order on first pass.

**Outputs written to disk:**

| Directory / File | Contents |
|---|---|
| `events/` | Per-participant events TSVs |
| `confounds/` | Per-participant selected confound matrices |
| `first_level/` | Per-participant t-maps (NIfTI) |
| `thresholded/` | Per-participant thresholded binary maps |
| `rois/` | Per-participant binary ROI masks |
| `group/` | Group-level t-map and fallback mask |
| `qc/` | QC figures |
| `summary_table.csv` | Final per-participant summary |


---
# 0. Setup

Imports, output directory creation, and all user-configurable parameters. **Edit this section before running anything else.** All subsequent sections read parameters from the variables defined here.

### 0.1 Imports

Loads all required libraries: `nilearn` (FirstLevelModel, image utilities, plotting), `nibabel`, `pandas`, `numpy`, `scipy.stats`, `matplotlib`, `pathlib`, `re`, `urllib`. No analysis here.

In [ ]:
# --- imports go here ---

### 0.2 Configuration

All paths and analysis parameters defined as constants.

**Paths:**
- `BIDS_DIR` — root of the fMRIPrep derivatives tree
- `EPRIME_DIR` — directory containing raw E-Prime `.txt` files
- `OUT_DIR` — root output directory for this localizer analysis
- `NEUROSYNTH_LOCAL` — local cache path for the ToM reverse-inference NIfTI
- `NEUROSYNTH_URL` — download URL for that map

**Subject list:**
- `SUBJECT_LIST` — list of 28 subject ID strings

**Acquisition parameters:**
- `TR` — repetition time in seconds
- `TASK_NAME` — BIDS task label (`EmotionalPain`)
- `SPACE` — MNI space label (`MNI152NLin2009cAsym_res-2`)

**GLM parameters:**
- `HRF_MODEL` — `'spm'`
- `DRIFT_MODEL` — `'cosine'`
- `NOISE_MODEL` — `'ar1'`
- `STANDARDIZE` — whether to z-score the BOLD signal before fitting

**Thresholding parameters:**
- `THRESHOLD_P` — uncorrected voxel-level p-value (0.001)
- `CLUSTER_K` — minimum cluster extent in voxels (10)
- `FALLBACK_MIN_VOXELS` — ROI voxel count below which fallback mask is used (10)
- `NEUROSYNTH_BINARIZE_THRESH` — posterior probability cutoff for binarizing the ToM map (e.g., 0.5)


In [ ]:
# --- path and parameter definitions go here ---

### 0.3 Create output directories

Creates all subdirectories (`events/`, `confounds/`, `first_level/`, `thresholded/`, `rois/`, `group/`, `qc/`) under `OUT_DIR` if they do not already exist. Safe to re-run.

In [ ]:
# --- output directory creation goes here ---

---
# 1. Parse E-Prime Behavioral Files

Reads raw E-Prime output (`.txt`, UTF-16 encoded) and produces one **events TSV** per participant in nilearn / BIDS format (`onset`, `duration`, `trial_type`).

**Relevant E-Prime fields:**

| E-Prime field | Role |
|---|---|
| `Wait4Scanner.OnsetTime` | Scanner trigger timestamp (ms) — defines time-zero |
| `ReadingList` | Condition: `1` = EP, `2` = PP |
| Story onset / offset fields | Used to compute `onset` and `duration` in seconds |

**Output per participant:** `events/{sub_id}_task-EmotionalPain_events.tsv`  
Columns: `onset` (s), `duration` (s), `trial_type` (`'EP'` or `'PP'`)


### 1.1 Helper: parse a single E-Prime file

Defines `parse_eprime(filepath) -> pd.DataFrame`. Steps:
1. Open with UTF-16 encoding.
2. Locate `Wait4Scanner.OnsetTime` → store as `t0` (ms).
3. Iterate over trial blocks; extract story onset, story offset, and `ReadingList` value.
4. Compute `onset = (story_onset_ms - t0) / 1000` and `duration = (story_offset_ms - story_onset_ms) / 1000`.
5. Map `ReadingList` 1 → `'EP'`, 2 → `'PP'`.
6. Return a DataFrame with columns `onset`, `duration`, `trial_type`, sorted by onset.


In [ ]:
# --- parse_eprime() function definition goes here ---

### 1.2 Run parser for all participants

Loops over `SUBJECT_LIST`, calls `parse_eprime()`, saves the result as a TSV. Prints a warning if:
- The E-Prime file is missing.
- The parsed table does not have exactly 10 rows (5 EP + 5 PP).
- Either condition is entirely absent.


In [ ]:
# --- parse + save events TSVs goes here ---

### 1.3 Sanity check: display parsed events

Concatenates events TSVs for all participants into a single DataFrame and displays summary statistics: onset range, duration range, trial counts per condition per participant. Flags any implausible values (negative onsets, durations outside a sensible range).


In [ ]:
# --- sanity check display goes here ---

---
# 2. Load & Select fMRIPrep Confounds

Loads `task-EmotionalPain_desc-confounds_timeseries.tsv` for each participant and extracts a lean confound matrix for the GLM.

**Confound selection:**

| Regressor | fMRIPrep column |
|---|---|
| Translation X/Y/Z | `trans_x`, `trans_y`, `trans_z` |
| Rotation X/Y/Z | `rot_x`, `rot_y`, `rot_z` |
| White matter signal | `white_matter` |
| CSF signal | `csf` |

NaNs in the first row (a fMRIPrep convention) are filled with 0.  
**To extend:** temporal derivatives or aCompCor regressors can be added here without touching any other section.

**Output per participant:** `confounds/{sub_id}_selected_confounds.tsv`


### 2.1 Helper: load and filter confounds for one participant

Defines `load_confounds(sub_id) -> pd.DataFrame` that reads the full fMRIPrep confounds TSV, selects the 8 columns above, and fills NaNs with 0.


In [ ]:
# --- load_confounds() function definition goes here ---

### 2.2 Run for all participants

Loops over subjects, calls `load_confounds()`, saves the selected matrix. Flags any subject whose confounds file is missing or whose matrix length does not match the BOLD run length (a mismatch would cause a GLM dimension error).


In [ ]:
# --- confound extraction loop goes here ---

---
# 3. First-Level GLM

Fits a mass-univariate GLM to each participant's BOLD time series using `nilearn.glm.first_level.FirstLevelModel`.

**Model specification:**
- **Regressors of interest:** EP and PP, convolved with SPM canonical HRF.
- **Drift:** cosine basis functions (removes low-frequency scanner drift).
- **Nuisance:** the 8 confound regressors from Section 2.
- **Noise model:** AR(1) — accounts for temporal autocorrelation in BOLD.
- `TR` passed from config.

**Inputs per participant:**
- `task-EmotionalPain_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz`
- Events TSV from Section 1
- Confound matrix from Section 2

> **Note:** Fitted model objects are held in memory (dict keyed by subject ID). This section must be re-run if the kernel restarts.


### 3.1 Fit GLM for all participants

Loops over subjects:
1. Resolves paths to the BOLD NIfTI, events TSV, and confound matrix.
2. Instantiates `FirstLevelModel` with parameters from config.
3. Calls `.fit()` with the BOLD image, events DataFrame, and confounds DataFrame.
4. Stores the fitted model in `models` dict.

A `try/except` catches missing files or fit failures and records them so the loop continues for remaining participants.


In [ ]:
# --- GLM fitting loop goes here ---

### 3.2 Spot-check design matrices

Plots the design matrix for 2–3 representative participants using `nilearn.plotting.plot_design_matrix`. Verify: EP and PP columns show expected HRF shape, cosine drift regressors are present, and nuisance columns appear at the right of the matrix.


In [ ]:
# --- design matrix spot-check plots go here ---

---
# 4. Compute EP > PP Contrast

Computes the contrast EP − PP for each participant and saves the t-statistic map.

The contrast is specified as the string `'EP - PP'` (nilearn resolves this to the correct column weights from the design matrix).

Degrees of freedom (df) are extracted from each fitted model and stored in a `participant_df` dict for use in Section 6, where the p-value → t-value conversion requires the actual df.

**Output per participant:** `first_level/{sub_id}_tmap_EP-PP.nii.gz`


### 4.1 Compute and save t-maps

Loops over fitted models, calls `model.compute_contrast('EP - PP', output_type='stat')`, saves each t-map as NIfTI, and records df in `participant_df`.


In [ ]:
# --- contrast computation + save loop goes here ---

---
# 5. Quality Control

Visual and quantitative inspection of the unthresholded t-maps **before** any thresholding. Do not skip on a first pass — outliers and artifacts are much easier to identify here than after the ROI pipeline has run.

**Checks performed:**
1. **Glass-brain grid** — t-map for every participant in a single figure.
2. **Voxel count bar plot** — number of voxels above a liberal fixed threshold (t > 2) per participant, with a horizontal line at `FALLBACK_MIN_VOXELS` to preview likely fallback cases.
3. **Summary statistics table** — max t, mean t (within EP > 0 voxels), voxel count above the liberal threshold.

All figures are saved to `qc/`.


### 5.1 Glass-brain grid: all participants

Produces a multi-panel figure (e.g., 4×7 for 28 participants) using `nilearn.plotting.plot_glass_brain` with a symmetric colormap (EP positive = warm, PP positive = cool). Saves to `qc/tmap_glassbrain_all.png`.


In [ ]:
# --- glass-brain grid plot goes here ---

### 5.2 Voxel count bar plot (liberal threshold)

Applies a fixed liberal threshold (t > 2.0, no mask) to each unthresholded t-map and plots the voxel count per participant as a horizontal bar chart sorted by count. A red dashed vertical line marks `FALLBACK_MIN_VOXELS`. Saves to `qc/voxel_counts_liberal.png`.


In [ ]:
# --- voxel count bar plot goes here ---

### 5.3 Per-participant summary statistics

Displays a DataFrame with: `subject_id`, `max_t`, `mean_t_positive`, `n_voxels_t2`. Flags participants with `max_t < 2` as potential data quality concerns.


In [ ]:
# --- summary stats table goes here ---

---
# 6. Threshold T-maps

Applies statistical thresholding + cluster extent filtering to produce a binary supra-threshold map per participant.

**Procedure:**
1. Retrieve the actual df from `participant_df` (Section 4).
2. Convert `THRESHOLD_P = 0.001` → t-value via `scipy.stats.t.ppf(1 - THRESHOLD_P, df)`. Using the actual df ensures the threshold is calibrated to each participant's GLM.
3. Binarize the t-map at this threshold (voxels with t ≥ threshold → 1, rest → 0).
4. Apply cluster extent filtering: label connected components (26-connectivity), discard clusters smaller than `CLUSTER_K` voxels.

**Output per participant:** `thresholded/{sub_id}_tmap_EP-PP_thresh.nii.gz`  
Also records participant-specific t-threshold and surviving voxel count in `thresh_results`.


### 6.1 Helper: threshold one t-map with cluster extent

Defines `threshold_tmap(tmap_img, t_thresh, cluster_k) -> Nifti1Image` that:
1. Converts the NIfTI to a numpy array and binarizes at `t_thresh`.
2. Uses `scipy.ndimage.label` to find connected components.
3. Zeros out components smaller than `cluster_k`.
4. Returns a new NIfTI with the surviving binary mask.


In [ ]:
# --- threshold_tmap() function definition goes here ---

### 6.2 Apply threshold for all participants

Loops over participants:
1. Looks up df from `participant_df`; computes participant-specific t-threshold.
2. Loads unthresholded t-map.
3. Calls `threshold_tmap()`.
4. Saves the thresholded binary mask.
5. Records `t_threshold` and `n_voxels_thresh` in `thresh_results`.


In [ ]:
# --- thresholding loop goes here ---

---
# 7. Anatomical Prior Mask

Prepares the Neurosynth reverse-inference map for "theory of mind" to constrain ROI definition to regions with independent functional evidence for mentalizing.

**Why this prior?** The EP > PP contrast is expected to engage the mentalizing / ToM network. Intersecting with a Neurosynth ToM map reduces false-positive voxels from non-specific activations (e.g., language, visual cortex).

**Steps:**
1. Download the NIfTI from `NEUROSYNTH_URL` if not already cached at `NEUROSYNTH_LOCAL`.
2. Binarize at `NEUROSYNTH_BINARIZE_THRESH` (e.g., posterior probability ≥ 0.5).
3. Resample to match the BOLD voxel grid using `nilearn.image.resample_to_img` with nearest-neighbor interpolation (preserves binary values).
4. Save and visualize.

**Output:** `group/neurosynth_ToM_mask_resampled.nii.gz` (binary, same voxel grid as BOLD)


### 7.1 Download Neurosynth ToM map

Checks whether `NEUROSYNTH_LOCAL` already exists; if not, downloads from `NEUROSYNTH_URL` using `urllib`. Prints the file size as a basic integrity check.


In [ ]:
# --- download / cache check goes here ---

### 7.2 Binarize, resample, save, and visualize

Loads the Neurosynth map with nibabel, applies the threshold to binarize, resamples to BOLD space, saves to `group/neurosynth_ToM_mask_resampled.nii.gz`. Plots the mask on the MNI152 template using `nilearn.plotting.plot_roi` so coverage can be verified before it is used for ROI definition.


In [ ]:
# --- binarize + resample + save + plot goes here ---

---
# 8. Intersect with Anatomical Prior → Individual Functional ROIs

Combines each participant's thresholded map (Section 6) with the ToM anatomical prior (Section 7) by voxel-wise AND to produce the final binary functional ROI.

**Inclusion rule:** A voxel enters the ROI only if it (a) survived statistical thresholding + cluster extent for this participant AND (b) falls within the Neurosynth ToM mask.

**Output per participant:** `rois/{sub_id}_EP-PP_roi.nii.gz`  
Records ROI voxel count and peak t-value in `roi_results` dict.


### 8.1 Intersect and save individual ROIs

Loops over participants:
1. Loads thresholded binary mask from `thresholded/`.
2. Multiplies element-wise with the resampled ToM mask (equivalent to logical AND for binary images).
3. Counts surviving voxels.
4. Extracts the peak t-value from the original (unthresholded) t-map within the ROI voxels.
5. Saves the binary ROI.
6. Records `{'n_voxels': ..., 'peak_t': ..., 'mask_type': 'individual'}` in `roi_results`.


In [ ]:
# --- intersection loop goes here ---

---
# 9. Fallback Rule: Group Mask for Low-Voxel Participants

Participants whose individual ROI has fewer than `FALLBACK_MIN_VOXELS` (10) voxels receive a group-level functional ROI instead, ensuring all participants have a usable mask for the main-task analysis.

**Fallback mask derivation:**
1. Identify fallback participants (`n_voxels < FALLBACK_MIN_VOXELS` in `roi_results`).
2. Compute a group EP > PP map by voxel-wise averaging of **all 28** unthresholded individual t-maps (not just fallback participants — using all data avoids selection bias).
3. Threshold the group t-map with the same p < 0.001 / cluster ≥ 10 procedure; use group-level df.
4. Intersect with the ToM anatomical prior.
5. Save as `group/group_EP-PP_fallback_roi.nii.gz`.
6. For each fallback participant: copy the group ROI to `rois/{sub_id}_EP-PP_roi.nii.gz`; update `mask_type` → `'fallback'` in `roi_results`.


### 9.1 Identify fallback participants

Iterates over `roi_results` and collects subject IDs where `n_voxels < FALLBACK_MIN_VOXELS`. Prints the count and list. If none: prints confirmation and skips the rest of this section.


In [ ]:
# --- fallback participant identification goes here ---

### 9.2 Compute group-level EP > PP map

Stacks all 28 unthresholded t-maps with `nilearn.image.concat_imgs` and computes a voxel-wise mean with `nilearn.image.math_img`. Applies `threshold_tmap()` (Section 6.1) using the group df. Intersects with the ToM mask. Saves `group/group_EP-PP_fallback_roi.nii.gz`. Visualizes the group ROI on a standard template.


In [ ]:
# --- group map computation + fallback ROI save goes here ---

### 9.3 Assign fallback masks and update results

For each fallback participant: writes the group ROI to `rois/{sub_id}_EP-PP_roi.nii.gz` (file copy, so `rois/` is self-contained). Updates `roi_results[sub_id]['mask_type']` to `'fallback'` and `roi_results[sub_id]['n_voxels']` to the group ROI voxel count.


In [ ]:
# --- fallback assignment goes here ---

---
# 10. Summary Table

Assembles and saves the final per-participant summary. This is the primary deliverable for downstream main-task analysis.

**Table columns:**

| Column | Description |
|---|---|
| `subject_id` | Participant identifier |
| `n_voxels_roi` | Voxels in the final ROI (individual or fallback) |
| `mask_type` | `'individual'` or `'fallback'` |
| `peak_t` | Peak t-value within the ROI (individual unthresholded map; NaN for fallback participants) |
| `t_threshold` | Participant-specific t-value used for thresholding |
| `roi_path` | Absolute path to the saved ROI NIfTI |

**Output:** `summary_table.csv`


### 10.1 Build and display summary DataFrame

Merges `roi_results` and `thresh_results` into a single `pd.DataFrame`, sorted by `subject_id`. Displays inline with fallback rows styled in a distinct color (e.g., light orange background via `DataFrame.style`) so they are immediately visible.


In [ ]:
# --- DataFrame construction + styled display goes here ---

### 10.2 Save summary CSV

Saves the DataFrame to `{OUT_DIR}/summary_table.csv` with `index=False`. Prints the full output path.


In [ ]:
# --- save CSV goes here ---

### 10.3 Final QC: ROI visualization grid

Plots all final ROI masks on the MNI152 template using `nilearn.plotting.plot_roi` in the same multi-panel grid format as the t-map glass-brain grid (Section 5.1). Fallback participants are labeled with an asterisk in the subplot title. Saves to `qc/roi_grid_all.png`.


In [ ]:
# --- final ROI visualization grid goes here ---